# AK(3) stable-proof lanes — one box per Colab runtime

Prove **AK(3) is stably AC-trivial** by finding any certified chain to the trivial presentation.
Open this notebook in up to **5 high-RAM CPU runtimes** and set `BOX` to a different value in each:

| BOX | what it runs | ~time |
|---|---|---|
| **D1** | Lane D (plateau elimination) at scale, **textbook** form: harvest 500k-node balls for 10 z=w stabilizations, Lemma-11-eliminate every visited state -> fresh 2-gen presentations ~stAC~ AK3, greedy-solve the 12,000 shortest @50k | ~8h |
| **D2** | same, **rep** form | ~8h |
| **D3** | Lane D with the FULL ~95-word bank, both forms (harvest 200k, solve @25k) | ~8-10h |
| **B**  | StableSolver (substitution+stabilize+eliminate in one search) @800k, g<=3/4 | ~3h |
| **C**  | dumb stabilization baselines n=3/4/5 @0.8-2M + MITM outward @2M with ball dumps | ~4h |

**Any `*** SOLVED ***` / verified cert = AK(3) is stably AC-trivial.** Everything is resumable:
if the runtime dies, just Run All again — finished work is skipped (JSONL on Drive).
When a box finishes (or in the morning), share the Drive folder `ak3_stable_proof/` back.

In [ ]:
# ==================== s1  CONFIG  (edit ONLY this cell) ====================
BOX          = "D1"        # <-- D1 | D2 | D3 | B | C  (one per runtime)
QUICK_TEST   = False       # True = ~2-min end-to-end base case FIRST; then set False and rerun s4
WORKERS      = None        # None = auto from CPU count

REPO_URL     = "https://github.com/Avi161/ACSolverX.git"
BRANCH       = "test/stable-ac-moves"
REPO_DIR     = "ACSolverX"
UPDATE_REPO  = True
MOUNT_DRIVE  = True
print(f"config: BOX={BOX} QUICK_TEST={QUICK_TEST}")

In [ ]:
# ==================== s2  SETUP  (clone / pull / install / mount) ====================
import os, subprocess, sys
def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-1500:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-1500:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    sh("pip -q install numpy numba")
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_DIR = f"/content/drive/MyDrive/ak3_stable_proof/{BOX}"
    else:
        OUT_DIR = f"/content/ak3_out/{BOX}"
    REPO_ROOT = os.path.abspath(REPO_DIR)
else:
    REPO_ROOT = os.path.abspath(".")
    OUT_DIR = os.path.abspath(f"./.scratch/colab_{BOX}")
os.makedirs(OUT_DIR, exist_ok=True)
NCPU = os.cpu_count()
print(f"repo={REPO_ROOT}  out={OUT_DIR}  cores={NCPU}")

In [ ]:
# ==================== s3  QUICK BASE CASE  (run once; ~2 min; must end 'complete') ====
if QUICK_TEST:
    script = os.path.join(REPO_ROOT, "experiments", "stable_ac", "ak3_stable_proof", "run_lanes.py")
    cmd = [sys.executable, script, "--box", BOX, "--out_dir", OUT_DIR + "_quick", "--quick"]
    if WORKERS: cmd += ["--workers", str(WORKERS)]
    print(" ".join(cmd)); subprocess.run(cmd, check=True)
    print("\nBASE CASE OK -> set QUICK_TEST=False and run s4")
else:
    print("QUICK_TEST=False -> skip (s4 is the real run)")

In [ ]:
# ==================== s4  RUN  (streams live; resumable after any disconnect) ==========
if not QUICK_TEST:
    script = os.path.join(REPO_ROOT, "experiments", "stable_ac", "ak3_stable_proof", "run_lanes.py")
    cmd = [sys.executable, script, "--box", BOX, "--out_dir", OUT_DIR]
    if WORKERS: cmd += ["--workers", str(WORKERS)]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("QUICK_TEST=True -> flip it to False first")

In [ ]:
# ==================== s5  TALLY  (run anytime) ====================
import glob, json
def tally_jsonl(path, key="solved"):
    n = s = 0
    for line in open(path):
        try: r = json.loads(line)
        except Exception: continue
        n += 1; s += int(bool(r.get(key)))
    return n, s

print(f"=== BOX {BOX} ===")
if BOX in ("D1", "D2", "D3"):
    sj = os.path.join(OUT_DIR, "laneD", "solve.jsonl")
    if os.path.exists(sj):
        n, s = tally_jsonl(sj)
        print(f"laneD solve: {n} candidates attempted, {s} SOLVED")
    for d in sorted(glob.glob(os.path.join(OUT_DIR, "laneD", "*.done"))):
        print(os.path.basename(d), open(d).read()[:160])
    certs = glob.glob(os.path.join(OUT_DIR, "certs", "laneD_*.json"))
    print(f"{len(certs)} laneD certificates:", *map(os.path.basename, certs), sep="\n  ")
else:
    for p in glob.glob(os.path.join(OUT_DIR, "box_*.jsonl")):
        print(os.path.basename(p))
        for line in open(p):
            r = json.loads(line)
            print("  ", json.dumps({k: r.get(k) for k in ("night_id","solved","hit","nodes","min_total_len","wall_s","peak_rss_mb","error")}))
print("\nIf anything says SOLVED / hit: share the Drive folder — that is the theorem.")